# AIProjectClient를 활용한 기본 RAG (Retrieval-Augmented Generation)

이 노트북에서는 다음을 사용하여 **기본적인 RAG 흐름**을 구현합니다:
- **`azure-ai-projects`** (AIProjectClient)
- **`azure-ai-inference`** (임베딩, ChatCompletions)
- **`azure-ai-search`** (벡터 또는 하이브리드 검색용)

주제는 **건강 & 피트니스** 🍏입니다.  
간단한 건강 팁 모음을 만들고, 이를 임베딩한 뒤, 검색 인덱스에 저장하고,  
사용자 쿼리를 통해 관련 팁을 검색하여 LLM에 전달해 최종 답변을 생성해보겠습니다.

> **안내문**: 이 노트북은 의료 조언을 제공하지 않습니다. 실제 건강 문제는 반드시 전문가와 상담하세요.

## RAG란 무엇인가요?
RAG(Retrieval-Augmented Generation)는 LLM(대규모 언어 모델)이  
자신의 데이터에서 검색한 **관련 텍스트 조각**을 사용해 최종 응답을 생성하는 기법입니다.  
이 방식은 모델의 답변을 실제 데이터에 기반하도록 하여 **환각(hallucination)** 현상을 줄이는 데 도움을 줍니다.

## 1. 설정  
라이브러리를 임포트하고, 환경 변수를 불러온 다음, `AIProjectClient`를 생성합니다.

> #### 이 노트북을 시작하기 전에 [2-embeddings.ipynb](2-embeddings.ipynb) 노트북을 먼저 완료하세요.


In [ ]:
import os
import time
import json
from dotenv import load_dotenv
from pathlib import Path

# azure-ai-projects
from azure.ai.projects import AIProjectClient
from azure.core.credentials import AzureKeyCredential

# We'll embed with azure-ai-inference
from azure.ai.inference import EmbeddingsClient, ChatCompletionsClient
from azure.ai.inference.models import UserMessage, SystemMessage

# For vector search or hybrid search
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

# Load environment variables
def _clean_env(value: str | None) -> str | None:
    if value is None:
        return None
    return value.strip().strip("\"").strip("'")

dotenv_candidates = [
    Path.cwd() / '.env',
    Path.cwd().parent / '.env',
    Path('.env'),
    Path('..') / '.env',
]
dotenv_path = next((p for p in dotenv_candidates if p.exists()), None)
if dotenv_path:
    load_dotenv(dotenv_path, override=False)
    print(f"✅ Loaded .env from: {dotenv_path}")
else:
    print("⚠️ .env file not found. Tried current/parent directories.")

endpoint = _clean_env(os.environ.get("ENDPOINT"))
api_key = _clean_env(os.environ.get("API_KEY"))
chat_model = _clean_env(os.environ.get("MODEL_NAME"))
embedding_model = _clean_env(os.environ.get("TEXT_EMBEDDING_MODEL"))
search_index_name = _clean_env(os.environ.get("SEARCH_INDEX_NAME"))

project_client = None
missing_keys = [
    name
    for name, value in {
        "ENDPOINT": endpoint,
        "API_KEY": api_key,
        "MODEL_NAME": chat_model,
        "TEXT_EMBEDDING_MODEL": embedding_model,
        "SEARCH_INDEX_NAME": search_index_name,
    }.items()
    if not value
]
if missing_keys:
    print(f"❌ Missing required env keys: {', '.join(missing_keys)}")
else:
    try:
        project_client = AIProjectClient(
            endpoint=endpoint,
            credential=AzureKeyCredential(api_key)
        )
        print("✅ AIProjectClient created successfully!")
    except Exception as e:
        print("❌ Error creating AIProjectClient:", e)

## 2. 샘플 건강 데이터 생성  
간단한 문서 조각 몇 개를 생성해보겠습니다.  
실제 시나리오에서는 CSV나 PDF 파일을 읽어 들여, 이를 청크로 나누고, 임베딩한 뒤 검색 인덱스에 저장할 수 있습니다.


In [ ]:
health_tips = [
    {
        "id": "doc1",
        "content": "Daily 30-minute walks help maintain a healthy weight and reduce stress.",
        "source": "General Fitness"
    },
    {
        "id": "doc2",
        "content": "Stay hydrated by drinking 8-10 cups of water per day.",
        "source": "General Fitness"
    },
    {
        "id": "doc3",
        "content": "Consistent sleep patterns (7-9 hours) improve muscle recovery.",
        "source": "General Fitness"
    },
    {
        "id": "doc4",
        "content": "For cardio endurance, try interval training like HIIT.",
        "source": "Workout Advice"
    },
    {
        "id": "doc5",
        "content": "Warm up with dynamic stretches before running to reduce injury risk.",
        "source": "Workout Advice"
    },
    {
        "id": "doc6",
        "content": "Balanced diets typically include protein, whole grains, fruits, vegetables, and healthy fats.",
        "source": "Nutrition"
    },
]
print("Created a small list of health tips.")

## 3.0. 인덱스 생성 또는 초기화  
Azure AI Search에서 벡터 필드를 생성할 때, **필드 정의(field definition)**에는  
`vector_search_profile` 속성이 포함되어 있어야 하며, 이는 벡터 검색 설정에서 정의된 프로필 이름과 일치해야 합니다.

HNSW 알고리즘 구성을 사용하여 벡터 인덱스를 생성하거나 초기화하는 헬퍼 함수를 정의하겠습니다.


In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchAlgorithmKind,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)

def create_healthtips_index(
        endpoint: str, api_key: str, index_name: str, 
        dimension: int = 1536 # if using text-embedding-3-small
        ):
    """Create or update a search index for health tips with vector search capability."""
    
    index_client = SearchIndexClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))
    
    # Try to delete existing index
    try:
        index_client.delete_index(index_name)
        print(f"Deleted existing index: {index_name}")
    except Exception:
        pass  # Index doesn't exist yet
        
    # Define vector search configuration
    vector_search = VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="myHnsw",
                kind=VectorSearchAlgorithmKind.HNSW,
                parameters=HnswParameters(
                    m=4,
                    ef_construction=400,
                    ef_search=500,
                    metric=VectorSearchAlgorithmMetric.COSINE
                )
            )
        ],
        profiles=[
            VectorSearchProfile(
                name="myHnswProfile",
                algorithm_configuration_name="myHnsw"
            )
        ]
    )
    
    # Define fields
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SimpleField(name="source", type=SearchFieldDataType.String),
        SearchField(
            name="embedding", 
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            vector_search_dimensions=dimension,
            vector_search_profile_name="myHnswProfile" 
        ),
    ]
    
    # Create index definition
    index_def = SearchIndex(
        name=index_name,
        fields=fields,
        vector_search=vector_search
    )
    
    # Create the index
    index_client.create_index(index_def)
    print(f"✅ Created or reset index: {index_name}")

## 3.1. 인덱스 생성 및 건강 팁 업로드

이제 건강 팁을 실제로 적용해보겠습니다. 다음과 같은 순서로 진행됩니다:

1. Azure AI Search에 **검색 연결**을 생성합니다.  
2. **벡터 검색 기능**이 포함된 인덱스를 생성합니다.  
3. 각 건강 팁에 대해 **임베딩을 생성**합니다.  
4. 임베딩과 함께 건강 팁을 **업로드**합니다.  

이 과정을 통해 나중에 검색할 수 있는 지식 기반을 구축하게 됩니다.  
AI 어시스턴트가 참조할 수 있는 '피트니스 도서관'을 만드는 것이라 생각하시면 됩니다!


In [ ]:
from azure.ai.projects.models import ConnectionType

search_endpoint = os.environ.get("SEARCH_ENDPOINT")
search_api_key = os.environ.get("SEARCH_API_KEY")

# Step 1: Get embeddings client and check embedding length
embeddings_client = EmbeddingsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(api_key),
    model=embedding_model
)
print("✅ Created embeddings client")

sample_doc = health_tips[0]
emb_response = embeddings_client.embed(
        input=[sample_doc["content"]]
    )
embedding_length = len(emb_response.data[0].embedding)
print(f"✅ Got embedding length: {embedding_length}")

# Step 2: Create the index
create_healthtips_index(
    endpoint=search_endpoint,
    api_key=search_api_key,
    index_name=search_index_name,
    dimension=embedding_length   # for text-embedding-3-large
)

# Step 4: Create search client for uploading documents
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=search_index_name,
    credential=AzureKeyCredential(search_api_key)
)
print("✅ Created search client")


# Step 5: Embed and upload documents
search_docs = []
for doc in health_tips:
    # Get embedding for document content
    emb_response = embeddings_client.embed(
        input=[doc["content"]]
    )
    emb_vec = emb_response.data[0].embedding
    
    # Create document with embedding
    search_docs.append({
        "id": doc["id"],
        "content": doc["content"],
        "source": doc["source"],
        "embedding": emb_vec,
    })

# Upload documents to index
result = search_client.upload_documents(documents=search_docs)
print(f"✅ Uploaded {len(search_docs)} documents to search index '{search_index_name}'")

## 4. 기본 RAG 흐름  
### 4.1. 검색(Retrieve)  
사용자가 질문을 입력하면, 다음 순서로 진행됩니다:  
1. 사용자 질문을 임베딩합니다.  
2. 해당 임베딩으로 벡터 인덱스를 검색하여 상위 문서를 가져옵니다.

### 4.2. 응답 생성(Generate answer)  
그 후, 검색된 문서들을 채팅 모델에 전달하여 응답을 생성합니다.

> 실제 시나리오에서는 더 정교한 청크 분할 및 요약 전략을 사용할 수 있습니다.  
> 이 예제에서는 단순한 형태로 진행합니다.


In [ ]:
from azure.search.documents.models import VectorizedQuery

def rag_chat(query: str, top_k: int = 3) -> str:
    # 1) Embed user query
    user_vec = embeddings_client.embed(
        input=[query]).data[0].embedding

    # 2) Vector search using VectorizedQuery
    vector_query = VectorizedQuery(
        vector=user_vec,
        k_nearest_neighbors=top_k,
        fields="embedding"
    )

    results = search_client.search(
        search_text="",  # Optional text query
        vector_queries=[vector_query],
        select=["content", "source"]  # Only retrieve fields we need
    )

    # gather the top docs
    top_docs_content = []
    for r in results:
        c = r["content"]
        s = r["source"]
        top_docs_content.append(f"Source: {s} => {c}")

    # 3) Chat with retrieved docs
    system_text = (
        "You are a health & fitness assistant.\n"
        "Answer user questions using ONLY the text from these docs.\n"
        "Docs:\n"
        + "\n".join(top_docs_content)
        + "\nIf unsure, say 'I'm not sure'.\n"
    )

    chat_client = ChatCompletionsClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(api_key),
        model=chat_model
    )

    response = chat_client.complete(
        messages=[
            SystemMessage(content=system_text),
            UserMessage(content=query)
        ],
    )

    return response.choices[0].message.content

## 5. 쿼리 실행해보기
바쁜 사람들을 위한 유산소 운동에 관한 질문을 해봅시다.


In [ ]:
user_query = "What's a good short cardio routine for me if I'm busy?"
answer = rag_chat(user_query)
print("🗣️ User Query:", user_query)
print("🤖 RAG Answer:", answer)

## 6. 마무리

이번 노트북에서는 다음을 포함한 **기본 RAG 파이프라인**을 시연했습니다:
- 문서를 **임베딩**하고 이를 **Azure AI Search**에 저장  
- 사용자 질문에 대해 상위 문서를 **검색(Retrieve)**  
- 검색된 문서들을 활용해 **채팅(Chat)** 수행  

이 파이프라인은 고급 청크 분할, 더욱 정교한 검색 전략, 품질 검사 등을 추가하여 확장할 수 있습니다.  
